# Survival Data Preprocessing — TCGA-BRCA
Loads `TCGA-BRCA.survival.tsv`, validates and saves `outcome.csv` used by all downstream notebooks.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# In Jupyter, Path().resolve() returns the working directory.
# The notebook is in Thesis_v3/01_Causal_feature_extraction/
# Go up until we find the folder containing "data/"
_cwd = Path().resolve()
BASE_DIR = next(
    p for p in [_cwd, *_cwd.parents]
    if (p / "data" / "TCGA-BRCA.survival.tsv").exists()
)

SURVIVAL_FILE = BASE_DIR / "data" / "TCGA-BRCA.survival.tsv"
OUTPUT_FILE   = BASE_DIR / "data" / "outcome.csv"

print(f"Base  : {BASE_DIR}")
print(f"Input : {SURVIVAL_FILE}")
print(f"Output: {OUTPUT_FILE}")
assert SURVIVAL_FILE.exists(), f"Not found: {SURVIVAL_FILE}"
print("Paths OK")


Base  : C:\Users\olegk\Desktop\Thesis_v3
Input : C:\Users\olegk\Desktop\Thesis_v3\data\TCGA-BRCA.survival.tsv
Output: C:\Users\olegk\Desktop\Thesis_v3\data\outcome.csv
Paths OK


In [3]:
surv = pd.read_csv(SURVIVAL_FILE, sep="\t", index_col=0)

print(f"Shape  : {surv.shape}")
print(f"Columns: {surv.columns.tolist()}")
print()
print(surv.head(5))


Shape  : (1232, 3)
Columns: ['OS.time', 'OS', '_PATIENT']

                  OS.time  OS      _PATIENT
sample                                     
TCGA-C8-A275-01A      1.0   0  TCGA-C8-A275
TCGA-AC-A7VC-01A      1.0   0  TCGA-AC-A7VC
TCGA-BH-A1F8-01A      1.0   1  TCGA-BH-A1F8
TCGA-BH-A1F8-11B      1.0   1  TCGA-BH-A1F8
TCGA-PL-A8LX-01A      5.0   0  TCGA-PL-A8LX


In [4]:
# Keep only OS and OS.time; drop _PATIENT (redundant with index)
outcome = surv[["OS.time", "OS"]].copy()

# Basic validation
print("=== Validation ===")
print(f"Samples        : {len(outcome):,}")
print(f"Missing OS.time: {outcome['OS.time'].isna().sum()}")
print(f"Missing OS     : {outcome['OS'].isna().sum()}")
print(f"OS values      : {sorted(outcome['OS'].dropna().unique().tolist())}")
print(f"OS.time units  : days (TCGA standard)")
print(f"OS.time range  : {outcome['OS.time'].min():.0f} – {outcome['OS.time'].max():.0f} days")
print(f"OS.time mean   : {outcome['OS.time'].mean():.1f} days  ({outcome['OS.time'].mean()/365.25:.1f} years)")
print(f"OS.time median : {outcome['OS.time'].median():.1f} days  ({outcome['OS.time'].median()/365.25:.1f} years)")
print(f"Events (OS=1)  : {int(outcome['OS'].sum())} ({outcome['OS'].mean()*100:.1f}%)")
print(f"Censored (OS=0): {int((outcome['OS']==0).sum())} ({(1-outcome['OS'].mean())*100:.1f}%)")


=== Validation ===
Samples        : 1,232
Missing OS.time: 0
Missing OS     : 0
OS values      : [0, 1]
OS.time units  : days (TCGA standard)
OS.time range  : 1 – 8605 days
OS.time mean   : 1273.2 days  (3.5 years)
OS.time median : 922.0 days  (2.5 years)
Events (OS=1)  : 201 (16.3%)
Censored (OS=0): 1031 (83.7%)


In [5]:
# Drop rows with missing survival data
n_before = len(outcome)
outcome  = outcome.dropna(subset=["OS.time", "OS"])
outcome  = outcome[outcome["OS.time"] > 0]   # remove zero follow-up
n_after  = len(outcome)

if n_before != n_after:
    print(f"Dropped {n_before - n_after} rows (missing or zero OS.time)")
else:
    print("No rows dropped")

# Sample ID format check
print(f"\nSample ID format (first 5):")
print(outcome.index[:5].tolist())
print(f"Example: TCGA-C8-A275-01A (barcode includes vial suffix)")


No rows dropped

Sample ID format (first 5):
['TCGA-C8-A275-01A', 'TCGA-AC-A7VC-01A', 'TCGA-BH-A1F8-01A', 'TCGA-BH-A1F8-11B', 'TCGA-PL-A8LX-01A']
Example: TCGA-C8-A275-01A (barcode includes vial suffix)


In [6]:
# 5-year survival summary (threshold = 1825 days)
THRESHOLD_5Y = 365.25 * 5

died_5y    = ((outcome["OS"] == 1) & (outcome["OS.time"] <= THRESHOLD_5Y)).sum()
alive_5y   = (outcome["OS.time"] >  THRESHOLD_5Y).sum()
censored   = ((outcome["OS"] == 0) & (outcome["OS.time"] <= THRESHOLD_5Y)).sum()

print("=== 5-year survival breakdown ===")
print(f"Events before 5 years  : {died_5y:,}  ({died_5y/len(outcome)*100:.1f}%)")
print(f"Survived past 5 years  : {alive_5y:,}  ({alive_5y/len(outcome)*100:.1f}%)")
print(f"Censored before 5 years: {censored:,}  ({censored/len(outcome)*100:.1f}%)")
print(f"Total                  : {len(outcome):,}")


=== 5-year survival breakdown ===
Events before 5 years  : 133  (10.8%)
Survived past 5 years  : 298  (24.2%)
Censored before 5 years: 801  (65.0%)
Total                  : 1,232


In [7]:
# Check sample ID format in RNA file and compare with survival IDs
import pandas as pd
from pathlib import Path

rna_raw = pd.read_csv(BASE_DIR / "data" / "TCGA-BRCA.star_counts.tsv",
                      sep="\t", index_col=0, nrows=0)  # header only, fast
rna_samples = set(rna_raw.columns.tolist())

print(f"RNA samples total        : {len(rna_samples)}")
print(f"Survival samples total   : {len(outcome)}")
print(f"Exact overlap            : {len(rna_samples & set(outcome.index))}")
print()

# TCGA barcode: TCGA-XX-XXXX-{sample_type}{vial}
# Sample type codes: 01-09 = tumor, 10-19 = normal, 20+ = control
# Keep only primary solid tumor (01x) and metastatic (06x)
vial_code = outcome.index.str[13:15]  # chars 13-14 = sample type
print("Sample type distribution in survival file:")
print(pd.Series(vial_code).value_counts().to_string())
print()

tumor_mask = vial_code.isin(["01", "06"])
outcome_tumor = outcome[tumor_mask].copy()

# If a patient has multiple tumor samples, keep the one present in RNA
# (or the first one if multiple match)
patient_id = outcome_tumor.index.str[:12]  # TCGA-XX-XXXX
duplicates = patient_id.duplicated(keep=False)
print(f"Duplicate patients (multiple tumor samples): {duplicates.sum()}")

# Prioritise samples that exist in RNA; otherwise keep first
in_rna     = outcome_tumor.index.isin(rna_samples)
keep_mask  = pd.Series(True, index=outcome_tumor.index)
for pid in patient_id[duplicates].unique():
    rows      = outcome_tumor[patient_id == pid]
    in_rna_rows = rows[rows.index.isin(rna_samples)]
    if len(in_rna_rows) >= 1:
        drop = rows.index.difference([in_rna_rows.index[0]])
    else:
        drop = rows.index[1:]
    keep_mask[drop] = False

outcome_clean = outcome_tumor[keep_mask].copy()
final_overlap = len(set(outcome_clean.index) & rna_samples)

print(f"After keeping primary tumor only : {len(outcome_clean)} samples")
print(f"Final overlap with RNA           : {final_overlap}")


RNA samples total        : 1226
Survival samples total   : 1232
Exact overlap            : 1203

Sample type distribution in survival file:
sample
01    1089
11     136
06       7

Duplicate patients (multiple tumor samples): 40
After keeping primary tumor only : 1076 samples
Final overlap with RNA           : 1073


In [8]:
outcome_clean.to_csv(OUTPUT_FILE)
print(f"Saved: {OUTPUT_FILE}")
print(f"Shape: {outcome_clean.shape}")
print(outcome_clean.head(3))


Saved: C:\Users\olegk\Desktop\Thesis_v3\data\outcome.csv
Shape: (1076, 2)
                  OS.time  OS
sample                       
TCGA-C8-A275-01A      1.0   0
TCGA-AC-A7VC-01A      1.0   0
TCGA-BH-A1F8-01A      1.0   1
